# Notebook 02: Hypothesis Testing and Experiment Evaluation

### Purpose
- Tests whether the Men's and Women's email campaigns drove meaningful changes in customer behavior, and translates statistical results into a business recommendation.

### Objectives
- Determine whether either email campaign had a statistically and practically significant effect on visits, conversions, and revenue.
- Assess robustness of spend results to outliers via winsorization, a necessary step given the heavy-tailed distribution characterized in notebook 01.
- Apply Bonferroni correction to account for multiple comparisons across 9 tests, and evaluate impact on the conclusions.
- Deliver a concrete recommendation: which campaign to roll out and what incremental revenue it implies per 10,000 customers.

In [1]:
import pandas as pd

In [5]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

display(hillstrom_df.head(), hillstrom_df.tail())

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


In [ ]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

comparisons = [("Mens E-Mail", "No E-Mail"), ("Mens E-Mail", "Womens E-Mail"), ("Womens E-Mail", "No E-Mail")]

visit_counts_df = hillstrom_df.groupby("segment")["visit"].agg(["sum", "count"])

results = []
for group1, group2 in comparisons:
    zstat, pvalue = proportions_ztest([visit_counts_df.loc[group1, "sum"], visit_counts_df.loc[group2, "sum"]],
                                      [visit_counts_df.loc[group1, "count"], visit_counts_df.loc[group2, "count"]])
    
    p1 = visit_counts_df.loc[group1, "sum"] / visit_counts_df.loc[group1, "count"]
    p2 = visit_counts_df.loc[group2, "sum"] / visit_counts_df.loc[group2, "count"]
    var1 = p1 * (1 - p1)
    var2 = p2 * (1 - p2)
    n1 = visit_counts_df.loc[group1, "count"]
    n2 = visit_counts_df.loc[group2, "count"]
    se = np.sqrt(var1 / n1 + var2 / n2)
    ci = (p1 - p2) + np.array([-1, 1]) * 1.96 * se
    ci = ci.round(4)

    results.append((group1, group2, zstat, pvalue, p1 - p2, ci))

results_df = pd.DataFrame(results, columns=["Group 1", "Group 2", "Z-Statistic", "P-Value", "Difference", "95% CI"])
results_df[["Z-Statistic", "Difference"]] = results_df[["Z-Statistic", "Difference"]].round(4)
results_df["P-Value"] = results_df["P-Value"].apply(lambda x: f"{x:.2e}")

display(results_df)

[0.07   0.0832]
[0.0243 0.0384]
[0.0389 0.0516]


,Group 1,Group 2,Z-Statistic,P-Value,Difference,95% CI
0,Mens E-Mail,No E-Mail,22.4860,5.69e-112,0.0766,"[0.07, 0.0832]"
1,Mens E-Mail,Womens E-Mail,8.6846,3.80e-18,0.0314,"[0.0243, 0.0384]"
2,Womens E-Mail,No E-Mail,13.9492,3.18e-44,0.0452,"[0.0389, 0.0516]"
